# Handling text with string methods

This document deals with handling text, a generally challenging task in data anaylses.  We will cover general `str` methods and introduce regular expressions (regex) to identify patterns in strings.

## Why is handling text important?

We will illustrate with several examples.

**Predicting stock returns**

|![text-financial](../img/text-financial-sentiment.png)|
|:--:|
|Truncated table of positive and negative text for predicting stock returns {cite:p}`glasserman2020choosing`|

**Parsing clinical notes**

|![text-clinical](../img/text-clinical-notes.png)|
|:--:|
|Text importance in long-term mortality prediction, truncated from table in {cite:p}`mahbub2022unstructured`|

**Untangling inspection violations**

In [ ]:
import pandas as pd
food = pd.read_csv('../data/inspections.csv')

In [ ]:
violations = food['Violations']

In [ ]:
violations[~violations.isna()].head()

In [ ]:
violations[0]

## Why is handling text difficult?

*(Class discussion)*

## Two primary goals of handling text

1. Canonicalization : transforming text of different representations into a standard form.

|![text_canon](../img/text-canon.png)|
|:--:|
|Example of canonization for joining tables with mismatched labels (DS 100).|

2. Extraction: extracting information into useful features.

Example: extracting dates, times, and other information from log files:

```
169.237.46.168 - - [26/Jan/2024:10:47:58 -0800] "GET /cs150/Winter24/ HTTP/1.1" 200 2585 "http://cs.northwestern.edu/courses"
```

```
day, month, year = "26", "Jan", "2024"
hour, minute, seconds = "10", "47", "58"
```

## Python string methods and `pandas` `str`
In base Python, manipulating strings is possible through various string methods.

In [27]:
s = "ABc,123,$%^&"
s

'ABc,123,$%^&'

In [29]:
s.lower()

'abc,123,$%^&'

In [30]:
s.upper()

'ABC,123,$%^&'

In [25]:
s.replace(',', ' ')

'ABC 123 $%^&'

In [20]:
s.split(',')

['ABC', '123', '$%^&']

In [22]:
'ABD' in s

False

In [31]:
len(s)

12

In [32]:
s[7:9]

',$'

**Issue?**
Although the string operations are useful (*recall our `apply()` function*), Python assumes we work with one string at a time.

Looping over each entry of a large dataset becomes slow.

In `pandas`, most of the operations are vectorized to perform on multiple entries simultaneously.

|       Operation      | Python (single string)  |   `pandas` (Series of strings) |
|:--------------------:|:------------------------|:-------------------------------|
| transformation       | `s.lower()`, `s.upper()`    | `ser.str.lower()`, `ser.str.upper()` |
| replacement/deletion | `s.replace(...)`            | `ser.str.replace(...)`             |
| split                | `s.split(...)`              | `ser.str.split(...)`               |
| substring            | `s[1:4]`                  | `ser.str[1:4]`                   |
| membership           | `'ab' in s`               | `ser.str.contains(...)`            |
| length               | `len(s)`                  | `ser.str.len()`                  |

## Practice: canonicalization 
Combine the following two tables into one, showing the county, state, and population as columns.

In [ ]:
import io
import pandas as pd

county_population = pd.read_csv(
    io.StringIO('''County,Population
    DeWitt,16798
    Lac Qui Parle,8067
    Lewis & Clark,55716
    St. John the Baptist,43044'''))

county_state = pd.read_csv(
    io.StringIO('''County,State
    De Witt County,IL
    Lac qui Parle County,MN
    Lewis and Clark County,MT
    St John the Baptist Parish,LS'''))

In [ ]:
def canon_text(s):
    result = (s.str.lower()
              .str.replace('&', 'and')
              .str.replace(' ', '')
              .str.replace('county', '')
              .str.replace('parish', '')
              .str.replace('.', '')
             )
    return result

In [ ]:
canon_text(county_population['County'])

In [41]:
canon_text(county_state['County'])

0              dewitt
1         lacquiparle
2       lewisandclark
3    stjohnthebaptist
Name: County, dtype: object

## Regular expressions
Many text data have "some" inherited structure, e.g., log files, concatenated violation codes.

## Practice: manipulating logs
**Starting with an example:**

Consider the two log entries below, how can we extract the dates and times, perhaps with `split()`?

```
169.237.46.168 - - [26/Jan/2024:10:47:58 -0800] "GET /cs150/Winter24/150.html HTTP/1.1" 200 2585 "http://cs.northwestern.edu/courses/"

193.205.203.3 - - [2/Feb/2023:17:23:6 -0800] "GET /iems394/Notes/dim.html HTTP/1.0" 404 302 "http://iems.northwestern.edu/academics/"
```

In [42]:
line1 = '169.237.46.168 - - [26/Jan/2024:10:47:58 -0800] "GET /cs150/Winter24/150.html HTTP/1.1" 200 2585 "http://cs.northwestern.edu/courses/"'
line2 = '193.205.203.3 - - [2/Feb/2023:17:23:6 -0800] "GET /iems394/Notes/dim.html HTTP/1.0" 404 302 "http://iems.northwestern.edu/academics/"'

In [50]:
def return_date_time(line):
    result = line.split('[')[1].split(']')[0]
    return result

In [51]:
return_date_time(line1)

'26/Jan/2024:10:47:58 -0800'

In [52]:
return_date_time(line2)

'2/Feb/2023:17:23:6 -0800'